In [1]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [8]:
import pandas as pd
messages = pd.read_csv('Spam (5).csv', sep=',', names = ['label','message'], skiprows=1)
messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [11]:
messages['length'] = messages['message'].apply(len)

In [12]:
messages.head()

,label,message,length
0,ham,"Go until jurong point, crazy.. Available only ...",111
1,ham,Ok lar... Joking wif u oni...,29
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,155
3,ham,U dun say so early hor... U c already then say...,49
4,ham,"Nah I don't think he goes to usf, he lives aro...",61


In [17]:
#NLP - Stopwords removing - Text pre-processing 
import string
from nltk.corpus import stopwords
stopwords.words('english') 
def text_process(mess):
    nopunc = [char for char in mess if char not in string.punctuation] 
    nopunc = ''.join(nopunc) 
    return [word for word in nopunc.split() if word.lower() not in stopwords.words('english')]

In [18]:
messages['message'].head().apply(text_process)

0    [Go, jurong, point, crazy, Available, bugis, n...
1                       [Ok, lar, Joking, wif, u, oni]
2    [Free, entry, 2, wkly, comp, win, FA, Cup, fin...
3        [U, dun, say, early, hor, U, c, already, say]
4    [Nah, dont, think, goes, usf, lives, around, t...
Name: message, dtype: object

In [19]:
messages.describe()

,length
count,5572.000000
mean,80.368988
std,59.926946
min,2.000000
25%,35.750000
50%,61.000000
75%,122.000000
max,910.000000


In [20]:
from sklearn.feature_extraction.text import CountVectorizer

In [23]:
bow_transformer=CountVectorizer(analyzer=text_process).fit(messages['message'])

In [24]:
print(len(bow_transformer.vocabulary_))

11422


In [25]:
messages_bow = bow_transformer.transform(messages['message'])

In [33]:



from sklearn.feature_extraction.text import TfidfTransformer

# Create TF-IDF transformer
tfidf_transformer = TfidfTransformer().fit(messages_bow)

# Convert the 4th message to Bag of Words
bow4 = bow_transformer.transform([messages['message'][3]])

# Transform it to TF-IDF
tfidf4 = tfidf_transformer.transform(bow4)

print(tfidf4.toarray())

[[0. 0. 0. ... 0. 0. 0.]]


In [37]:
messages_tfidf = tfidf_transformer.transform(messages_bow) 
print(messages_tfidf.shape)
from sklearn.naive_bayes import MultinomialNB  
spam_detect_model = MultinomialNB() # model creation 
spam_detect_model.fit(messages_tfidf, messages['label']) # training 
from sklearn.metrics import classification_report  # model evl.  

all_predictions =  spam_detect_model.predict(messages_tfidf) 
print(classification_report(messages['label'], all_predictions))
print(all_predictions) # model testing

(5572, 11422)
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99      4825
        spam       1.00      0.85      0.92       747

    accuracy                           0.98      5572
   macro avg       0.99      0.92      0.95      5572
weighted avg       0.98      0.98      0.98      5572

['ham' 'ham' 'spam' ... 'ham' 'ham' 'ham']


In [41]:



msg4 = messages['message'][3]

bow4 = bow_transformer.transform([msg4])

tfidf4 = tfidf_transformer.transform(bow4)

print(tfidf4.toarray())
print(bow4)
print(bow4.shape)

print("Predicted:", spam_detect_model.predict(tfidf4)[0])
print("Expected:", messages['label'][3])
                                    


[[0. 0. 0. ... 0. 0. 0.]]
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 7 stored elements and shape (1, 11422)>
  Coords	Values
  (0, 4066)	2
  (0, 4627)	1
  (0, 5258)	1
  (0, 6201)	1
  (0, 6219)	1
  (0, 7183)	1
  (0, 9551)	2
(1, 11422)
Predicted: ham
Expected: ham


In [42]:
print(messages['message'][3])

U dun say so early hor... U c already then say...


In [43]:
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing import sequence
tf.random.set_seed(7) 

In [49]:
top_words = 500
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words= top_words)


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 9s 1us/step


In [50]:
max_review_length = 500
X_train = sequence.pad_sequences(X_train, maxlen= max_review_length)
X_test = sequence.pad_sequences(X_test, maxlen = max_review_length)

In [54]:
embedding_vector_length = 16
model = Sequential()
model.add(Embedding(top_words, embedding_vector_length, input_shape=(max_review_length,)))
model.add(SimpleRNN(100))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss = 'binary_crossentropy', optimizer='adam' , metrics=['accuracy'])
print(model.summary())

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 500, 16)        │         8,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 100)            │        11,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │           101 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,801 (77.35 KB)

 Trainable params: 19,801 (77.35 KB)

 Non-trainable params: 0 (0.00 B)

None


In [57]:
model.fit(X_train, y_train, validation_data= (X_test,y_test), epochs = 3, batch_size= 64)

Epoch 1/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 53s 124ms/step - accuracy: 0.5326 - loss: 0.6896 - val_accuracy: 0.5995 - val_loss: 0.6654
Epoch 2/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 46s 116ms/step - accuracy: 0.6100 - loss: 0.6514 - val_accuracy: 0.5677 - val_loss: 0.6702
Epoch 3/3
391/391 ━━━━━━━━━━━━━━━━━━━━ 47s 120ms/step - accuracy: 0.6016 - loss: 0.6500 - val_accuracy: 0.6377 - val_loss: 0.6221


In [58]:
scores = model.evaluate(X_test, y_test, verbose= 0)
print("Accuracy: %.2f%%" % (scores [1]*100))

Accuracy: 63.77%
